# Séance 7 – Classes et instances (2)
### Lab : attributs de classe, méthodes spéciales et propriétés

| | |
|---|---|
| **Durée estimée** | 60 min (2 × 30 min) + bonus |
| **Projet fil rouge** | GestParc – étape 5 : Equipment complète, Employee et Inventory |

**Objectifs**
- Utiliser attributs de classe, @classmethod et @staticmethod
- Implémenter les méthodes spéciales usuelles
- Encapsuler avec _, __ et @property

**Mode d'emploi**
- Exécuter une cellule : **Maj + Entrée** ; en cas de comportement étrange : *Redémarrer le noyau et tout exécuter*.
- Complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Vérification** qui suit : aucune erreur = réponse correcte.
- Difficulté : ⭐ facile · ⭐⭐ intermédiaire · ⭐⭐⭐ avancé · Indices dans les blocs 💡 repliables.

---
## Partie 1 – Attributs de classe, méthodes de classe et statiques

> **Rappel**
> - Attribut de classe : défini dans le corps de la classe, partagé ; modifié via `NomClasse.attr`
> - `@classmethod def f(cls, ...)` : reçoit la classe → constructeurs alternatifs `return cls(...)`
> - `@staticmethod def f(...)` : fonction utilitaire sans `self` ni `cls`

### Exemple guidé : compteur d'instances et constructeur alternatif

In [ ]:
class Employee:
    """Employé de l'entreprise."""

    DOMAIN = "entreprise.tn"
    headcount = 0

    def __init__(self, first_name: str, last_name: str) -> None:
        self.first_name = first_name
        self.last_name = last_name
        Employee.headcount += 1

    @classmethod
    def from_email(cls, email: str) -> "Employee":
        local = email.split("@")[0]
        first, last = local.split(".")
        return cls(first.capitalize(), last.capitalize())

    @staticmethod
    def is_professional(email: str) -> bool:
        return email.endswith("@" + Employee.DOMAIN)


karim = Employee.from_email("karim.trabelsi@entreprise.tn")
print(karim.first_name, karim.last_name, Employee.headcount)
print(Employee.is_professional("karim@gmail.com"))

### Exercice 1 ⭐ – Numérotation automatique des tickets

Écrivez la classe `Ticket(title: str)` :
- un attribut de classe `_next_number` (initialement `1`) ;
- chaque nouveau ticket reçoit un attribut `ticket_id` de la forme `"INC-0001"`, `"INC-0002"`… ;
- une méthode de classe `reset_counter(cls, start: int = 1)` remet le compteur à `start`.

In [ ]:
class Ticket:
    """Ticket numéroté automatiquement."""
    # TODO


t1, t2 = Ticket("VPN lent"), Ticket("Écran noir")
print(t1.ticket_id, t2.ticket_id)

<details><summary>💡 Indice</summary>

Dans `__init__`, lisez et incrémentez `Ticket._next_number` (et non `self._next_number += 1`, qui créerait un attribut d'instance).

</details>

In [ ]:
# ✅ Vérification
Ticket.reset_counter()
a, b, c = Ticket("A"), Ticket("B"), Ticket("C")
assert (a.ticket_id, b.ticket_id, c.ticket_id) == ("INC-0001", "INC-0002", "INC-0003")
Ticket.reset_counter(500)
assert Ticket("D").ticket_id == "INC-0500"
assert "_next_number" not in vars(a), "le compteur doit rester un attribut de classe"
print("✅ Exercice 1 réussi")

### Exercice 2 ⭐⭐ – Corriger un attribut de classe partagé

La classe `Playbook` ci-dessous a deux bugs : les tâches et les hôtes sont partagés entre tous les playbooks.
Corrigez-la **sans changer son interface** (mêmes méthodes, mêmes attributs).

In [ ]:
class Playbook:
    """Playbook d'automatisation (liste de tâches appliquées à des hôtes)."""

    DEFAULT_FORKS = 5
    tasks = []
    hosts = set()

    def __init__(self, name: str) -> None:
        self.name = name

    def add_task(self, task: str) -> None:
        self.tasks.append(task)

    def target(self, *hosts: str) -> None:
        self.hosts.update(hosts)


web = Playbook("web")
web.add_task("installer nginx")
web.target("srv-web-01", "srv-web-02")
db = Playbook("db")
print(web.tasks, db.tasks, db.hosts, Playbook.DEFAULT_FORKS)

<details><summary>💡 Indice</summary>

`DEFAULT_FORKS` est une constante : elle peut rester attribut de classe. Les collections propres à chaque playbook se créent dans `__init__`.

</details>

In [ ]:
# ✅ Vérification
p1, p2 = Playbook("p1"), Playbook("p2")
p1.add_task("t1")
p1.target("h1")
assert p2.tasks == [] and p2.hosts == set()
assert p1.tasks == ["t1"] and p1.hosts == {"h1"}
assert Playbook.DEFAULT_FORKS == 5
print("✅ Exercice 2 réussi")

### Exercice 3 ⭐⭐ – Constructeur alternatif et validation

Complétez la classe `Server` :
- `is_valid_ip(ip)` (**statique**) : 4 nombres de 0 à 255 séparés par des points ;
- `from_inventory_line(line)` (**de classe**) : crée un serveur à partir de `"hostname;ip;cpu;ram_gb"` (espaces et retour à la ligne tolérés) ;
- `from_dict(data)` (**de classe**) : à partir d'un dictionnaire avec les clés `hostname`, `ip`, `cpu`, `ram_gb` ;
- l'attribut `valid` (booléen) indique si l'IP est valide (calculé dans `__init__` avec `is_valid_ip`).

In [ ]:
class Server:
    """Serveur du parc."""

    def __init__(self, hostname: str, ip: str, cpu: int, ram_gb: int) -> None:
        self.hostname = hostname
        self.ip = ip
        self.cpu = cpu
        self.ram_gb = ram_gb
        # TODO : attribut valid

    # TODO : is_valid_ip, from_inventory_line, from_dict


lines = ["srv-web-01; 10.0.0.5 ;8;32\n", "srv-db-01;10.0.0.300;16;128\n"]
servers = [Server.from_inventory_line(line) for line in lines]
print([(s.hostname, s.ip, s.valid) for s in servers])

<details><summary>💡 Indice</summary>

`[f.strip() for f in line.split(";")]` nettoie chaque champ. Une méthode de classe se termine par `return cls(...)`.

</details>

In [ ]:
# ✅ Vérification
assert Server.is_valid_ip("192.168.1.10") and not Server.is_valid_ip("10.0.0") and not Server.is_valid_ip("10.0.0.x")
s = Server.from_inventory_line("srv-app-01 ; 10.0.1.7 ; 4 ; 16\n")
assert (s.hostname, s.ip, s.cpu, s.ram_gb, s.valid) == ("srv-app-01", "10.0.1.7", 4, 16, True)
d = Server.from_dict({"hostname": "srv-x", "ip": "300.1.1.1", "cpu": 2, "ram_gb": 4})
assert isinstance(d, Server) and d.valid is False
print("✅ Exercice 3 réussi")

### Exercice 4 ⭐⭐⭐ – La classe Duration

Écrivez la classe `Duration`, qui stocke une durée en **minutes** (attribut `minutes`) :
- `from_hours(hours: float)` : constructeur alternatif ;
- `parse(text: str)` : statique, convertit `"1h30"`, `"45min"`, `"2h"` en nombre de minutes (`int`) ;
- `from_text(text: str)` : constructeur alternatif qui utilise `parse` ;
- `to_text()` : `"1h30"`, `"45min"`, `"2h"` (format inverse ; `"0min"` pour une durée nulle).

In [ ]:
class Duration:
    """Durée exprimée en minutes."""

    def __init__(self, minutes: int) -> None:
        self.minutes = minutes

    # TODO : from_hours, parse, from_text, to_text


print(Duration.from_text("1h30").minutes, Duration.from_hours(2.25).to_text(), Duration(45).to_text())

<details><summary>💡 Indice</summary>

`"1h30".partition("h")` renvoie `('1', 'h', '30')`. `divmod(95, 60)` renvoie `(1, 35)`.

</details>

In [ ]:
# ✅ Vérification
assert Duration.parse("1h30") == 90 and Duration.parse("45min") == 45 and Duration.parse(" 2H ") == 120
assert Duration.from_text("1h05").minutes == 65
assert Duration.from_hours(2.25).minutes == 135
assert Duration(90).to_text() == "1h30" and Duration(120).to_text() == "2h"
assert Duration(45).to_text() == "45min" and Duration(0).to_text() == "0min" and Duration(65).to_text() == "1h05"
print("✅ Exercice 4 réussi")

---
## Partie 2 – Méthodes spéciales et encapsulation

> **Rappel**
> - `__str__` (lisible), `__repr__` (non ambigu) · `__eq__`, `__lt__` (comparaison, tri)
> - `__len__`, `__contains__`, `__getitem__` (collections) · `__add__`, `__sub__`, `__mul__` (opérateurs)
> - `_attribut` : interne par convention · `__attribut` : *name mangling*
> - `@property` (lecture) + `@nom.setter` (écriture contrôlée) ; le setter écrit dans `self._nom`

### Exemple guidé : une file d'attente de tickets

In [ ]:
class TicketQueue:
    """File d'attente de tickets, utilisable comme une collection."""

    def __init__(self, name: str) -> None:
        self.name = name
        self._tickets: list[str] = []

    def push(self, ticket: str) -> None:
        self._tickets.append(ticket)

    def __len__(self) -> int:
        return len(self._tickets)

    def __contains__(self, ticket: str) -> bool:
        return ticket in self._tickets

    def __getitem__(self, index: int) -> str:
        return self._tickets[index]

    def __str__(self) -> str:
        return f"File {self.name} : {len(self)} ticket(s)"


queue = TicketQueue("Support N1")
queue.push("INC-0001")
queue.push("INC-0002")
print(queue, "INC-0002" in queue, queue[0], queue[-1])
for ticket in queue:           # __getitem__ suffit pour parcourir l'objet
    print("-", ticket)

### Exercice 5 ⭐ – __str__ et __repr__

Ajoutez à la classe `VirtualMachine` :
- `__str__` → `"vm-web-01 (2 vCPU, 4 Go) – running"` ;
- `__repr__` → `"VirtualMachine('vm-web-01', cpu=2, ram_gb=4)"`.

In [ ]:
class VirtualMachine:
    """Machine virtuelle."""

    def __init__(self, name: str, cpu: int = 2, ram_gb: int = 4) -> None:
        self.name = name
        self.cpu = cpu
        self.ram_gb = ram_gb
        self.state = "running"

    # TODO


vm = VirtualMachine("vm-web-01")
print(vm)
print([vm])

<details><summary>💡 Indice</summary>

`{self.name!r}` ajoute les guillemets autour du nom.

</details>

In [ ]:
# ✅ Vérification
vm = VirtualMachine("vm-db-01", cpu=8, ram_gb=32)
assert str(vm) == "vm-db-01 (8 vCPU, 32 Go) – running"
assert repr(vm) == "VirtualMachine('vm-db-01', cpu=8, ram_gb=32)"
assert f"{vm}" == str(vm)
print("✅ Exercice 5 réussi")

### Exercice 6 ⭐⭐ – Trier des versions de logiciels

Écrivez la classe `SoftwareVersion(text: str)` (`"3.12.1"`, `"2.4"`) :
- attribut `parts` : tuple d'entiers ;
- `__eq__` : deux versions sont égales si leurs `parts` le sont (`"2.4"` est égale à `"2.4.0"` : complétez par des zéros jusqu'à 3 nombres) ;
- `__lt__` : ordre numérique ;
- `__str__` : le texte d'origine ; `__repr__` : `SoftwareVersion('3.12.1')`.

Puis calculez `outdated` : les serveurs dont la version d'OpenSSL est **strictement inférieure** à `minimum`.

In [ ]:
class SoftwareVersion:
    """Numéro de version comparable."""
    # TODO


openssl = {"srv-web-01": "3.0.13", "srv-web-02": "3.2.1", "srv-db-01": "1.1.1", "srv-app-01": "3.0.2"}
minimum = SoftwareVersion("3.0.13")
outdated = ...

<details><summary>💡 Indice</summary>

`sorted()` n'a besoin que de `__lt__`. Comparer des chaînes donnerait `"3.10" < "3.9"` : faux numériquement !

</details>

In [ ]:
# ✅ Vérification
assert SoftwareVersion("2.4") == SoftwareVersion("2.4.0") and SoftwareVersion("3.9") != SoftwareVersion("3.10")
assert SoftwareVersion("3.9.18") < SoftwareVersion("3.12.1")
assert SoftwareVersion("1.0") != "1.0"
assert [str(v) for v in sorted([SoftwareVersion("3.13"), SoftwareVersion("3.9.1"), SoftwareVersion("3.12.0")])] == ["3.9.1", "3.12.0", "3.13"]
assert repr(SoftwareVersion("3.12.1")) == "SoftwareVersion('3.12.1')"
assert outdated == ["srv-app-01", "srv-db-01"]
print("✅ Exercice 6 réussi")

### Exercice 7 ⭐⭐ – La classe Money

Écrivez la classe `Money(millimes: int)` (1 DT = 1 000 millimes) :
- `from_dinars(amount: float)` (de classe) : `Money.from_dinars(12.5)` → 12 500 millimes ;
- `__add__` et `__sub__` entre deux `Money` ; `__mul__` par un entier (quantité) ;
- `__eq__`, `__lt__` ;
- `__str__` → `"12.500 DT"` ; `__repr__` → `"Money(12500)"`.

Calculez ensuite `total` (le prix total de la commande) et `remaining` (budget restant).

In [ ]:
class Money:
    """Montant en dinars tunisiens, stocké en millimes."""
    # TODO


order = [("Clavier", Money.from_dinars(45.9), 10), ("Écran 27 pouces", Money.from_dinars(649.0), 4), ("Câble HDMI", Money(12_350), 6)]
total = ...
budget = Money.from_dinars(3500)
remaining = ...
print(total, remaining, total < budget)

<details><summary>💡 Indice</summary>

Chaque opérateur renvoie un **nouvel** objet `Money`. Pour accumuler : `total = total + ...`.

</details>

In [ ]:
# ✅ Vérification
assert Money.from_dinars(12.5) == Money(12500) and repr(Money(12500)) == "Money(12500)"
assert str(Money(12500)) == "12.500 DT"
assert Money(1000) + Money(500) == Money(1500) and Money(1000) - Money(1500) == Money(-500)
assert Money(1500) * 3 == Money(4500) and Money(1) < Money(2)
assert total == Money(3129100) and str(remaining) == "370.900 DT"
print("✅ Exercice 7 réussi")

### Exercice 8 ⭐⭐ – Un pool de serveurs

Écrivez la classe `ServerPool(name: str)` qui contient des noms de serveurs (liste interne `_servers`) :
- `add(*hostnames)` : ajoute des serveurs en ignorant les doublons ;
- `__len__`, `__contains__`, `__getitem__` (accès par index) ;
- `__add__` : `pool_a + pool_b` renvoie un **nouveau** pool nommé `"a+b"` contenant les serveurs des deux (sans doublon, ordre conservé) ;
- `__str__` → `"web: 3 serveur(s)"`.

In [ ]:
class ServerPool:
    """Groupe de serveurs."""
    # TODO


web = ServerPool("web")
web.add("srv-web-01", "srv-web-02", "srv-web-01")
db = ServerPool("db")
db.add("srv-db-01", "srv-web-02")
everything = web + db
print(web, db, everything, list(everything))

<details><summary>💡 Indice</summary>

Dans `__add__`, créez un nouveau `ServerPool` et réutilisez `add` avec la décompression `*`.

</details>

In [ ]:
# ✅ Vérification
assert len(web) == 2 and "srv-web-02" in web and "srv-db-01" not in web
assert web[0] == "srv-web-01" and web[-1] == "srv-web-02"
assert str(everything) == "web+db: 3 serveur(s)" and list(everything) == ["srv-web-01", "srv-web-02", "srv-db-01"]
assert len(web) == 2, "web + db ne doit pas modifier web"
print("✅ Exercice 8 réussi")

### Exercice 9 ⭐⭐⭐ – Un onduleur avec propriétés

Écrivez la classe `UPS(name: str, capacity_va: int, charge: float = 100.0)` (onduleur) :
- propriété `charge` (en %) : le setter **ramène** la valeur dans [0, 100] (`120` → `100`, `-5` → `0`) ;
- propriété en **lecture seule** `status` : `"OK"` (≥ 50), `"LOW"` (≥ 20), `"CRITICAL"` (< 20) ;
- propriété en lecture seule `capacity_va` (stockée dans `__capacity_va`, attribut privé) ;
- méthode `discharge(minutes: int, rate: float = 1.5)` : la charge baisse de `rate` % par minute.

In [ ]:
class UPS:
    """Onduleur de salle serveur."""
    # TODO


ups = UPS("ups-salle-a", 3000)
ups.discharge(40)
print(ups.charge, ups.status)
ups.discharge(30)
print(ups.charge, ups.status)

<details><summary>💡 Indice</summary>

Dans `__init__`, écrivez `self.charge = charge` : le setter s'applique dès la création. `discharge` réutilise aussi le setter.

</details>

In [ ]:
# ✅ Vérification
u = UPS("test", 1500, charge=120)
assert u.charge == 100.0 and u.status == "OK"
u.charge = -5
assert u.charge == 0.0 and u.status == "CRITICAL"
u.charge = 35
assert u.status == "LOW"
u.discharge(10, rate=2)
assert u.charge == 15.0 and u.status == "CRITICAL"
assert u.capacity_va == 1500
try:
    u.status = "OK"
    raise AssertionError("status doit être en lecture seule")
except AttributeError:
    pass
try:
    u.capacity_va = 99
    raise AssertionError("capacity_va doit être en lecture seule")
except AttributeError:
    pass
assert "_UPS__capacity_va" in vars(u)
print("✅ Exercice 9 réussi")

---
### Exercice de synthèse ⭐⭐⭐ – La classe IPv4Address

Écrivez la classe `IPv4Address`, qui stocke l'adresse sous forme d'un entier (attribut `value`, entre 0 et 2³² − 1) :
- `is_valid(text)` (**statique**) et `from_string(text)` (**de classe**) ;
- `__str__` → `"192.168.1.10"` ; `__repr__` → `"IPv4Address('192.168.1.10')"` ;
- `__eq__`, `__lt__` (ordre numérique, donc `10.0.0.9 < 10.0.0.10`) ;
- `__add__(n: int)` → l'adresse située `n` positions plus loin ;
- propriété `is_private` : vraie pour `10.0.0.0/8`, `172.16.0.0/12` (172.16 à 172.31) et `192.168.0.0/16`.

Utilisez-la pour trier `addresses` et générer `pool` : les 5 adresses à partir de `192.168.10.250`.

In [ ]:
class IPv4Address:
    """Adresse IPv4 stockée sous forme d'entier."""
    # TODO


addresses = [IPv4Address.from_string(t) for t in ["10.0.0.10", "10.0.0.9", "192.168.1.1", "8.8.8.8"]]
print(sorted(addresses))
start = IPv4Address.from_string("192.168.10.250")
pool = ...

<details><summary>💡 Indice</summary>

- a.b.c.d → `a × 256³ + b × 256² + c × 256 + d`.
- Inverse : `v // 256**3`, `v // 256**2 % 256`, `v // 256 % 256`, `v % 256`.
- Pour aller plus loin, la bibliothèque standard propose le module `ipaddress`… qui fait exactement cela !

</details>

In [ ]:
# ✅ Vérification
ip = IPv4Address.from_string("192.168.1.10")
assert ip.value == 3232235786 and str(ip) == "192.168.1.10" and repr(ip) == "IPv4Address('192.168.1.10')"
assert IPv4Address.is_valid("10.0.0.1") and not IPv4Address.is_valid("10.0.0.256")
assert IPv4Address.from_string("10.0.0.9") < IPv4Address.from_string("10.0.0.10")
assert IPv4Address.from_string("10.0.0.9") == IPv4Address.from_string("10.0.0.9")
assert [str(a) for a in sorted(addresses)] == ["8.8.8.8", "10.0.0.9", "10.0.0.10", "192.168.1.1"]
assert pool == ["192.168.10.250", "192.168.10.251", "192.168.10.252", "192.168.10.253", "192.168.10.254"]
assert str(IPv4Address.from_string("10.0.0.255") + 1) == "10.0.1.0"
assert [IPv4Address.from_string(t).is_private for t in ["172.20.1.1", "172.32.0.1", "8.8.8.8", "10.1.2.3"]] == [True, False, False, True]
print("✅ Exercice de synthèse réussi")

---
## Projet fil rouge – étape 5 : `Equipment` complète, `Employee` et `Inventory`

![UML étape 5](img/s07_gestparc.svg)

**À faire :**
1. **`Equipment`** :
   - attributs de classe `CATEGORIES` et `count` (nombre d'équipements créés) ;
   - `is_valid_tag(tag)` (statique) : 2 lettres, un tiret, 3 chiffres ;
   - `from_dict(data)` (de classe) et `to_dict()` ;
   - propriété `price` : le setter refuse un prix négatif (message, valeur inchangée) ;
   - propriété en lecture seule `assigned_to` (attribut interne `_assigned_to`), modifiée uniquement par `assign` / `unassign` ;
   - `__str__` (ancien `describe`), `__repr__`, `__eq__` (même numéro), `__lt__` (ordre des numéros).
2. **`Employee`** : `first_name`, `last_name`, `department`, propriété `full_name`, `__str__`.
3. **`Inventory`** : `add` (refuse catégorie inconnue, numéro invalide ou déjà présent), `find`, `assign(tag, employee)`, `stats`, `total_value`, `available`, `older_than`, `__len__`, `__contains__` (numéro **ou** équipement), `__getitem__` (index).

In [ ]:
class Equipment:
    """Équipement du parc informatique."""
    # TODO : reprendre la classe de l'étape 4 et la compléter


class Employee:
    """Employé pouvant recevoir des équipements."""
    # TODO


class Inventory:
    """Inventaire du parc informatique."""
    # TODO


inventory = Inventory()
inventory.add(Equipment("SV-001", "server", "HPE", "ProLiant DL380", 2019, 18500))
inventory.add(Equipment("LT-001", "laptop", "Dell", "Latitude 5440", 2023, 3200))
inventory.add(Equipment.from_dict({"asset_tag": "PR-001", "category": "printer", "brand": "HP",
                                   "model": "LaserJet M404", "purchase_year": 2021, "price": 950}))
amira = Employee("Amira", "Ben Salah", "DSI")
inventory.assign("lt-001", amira)
for equipment in sorted(inventory):
    print(equipment)
print(len(inventory), "LT-001" in inventory, inventory.stats(), f"{inventory.total_value():,.2f} DT")

In [ ]:
# ✅ Vérification
before = Equipment.count
e = Equipment("lt-050", "laptop", "Asus", "ExpertBook", 2024, 2500)
assert Equipment.count == before + 1
assert Equipment.is_valid_tag("LT-050") and not Equipment.is_valid_tag("LT-50") and not Equipment.is_valid_tag("L1-050")
e.price = -10
assert e.price == 2500.0
try:
    e.assigned_to = "X"
    raise AssertionError("assigned_to doit être en lecture seule")
except AttributeError:
    pass
assert Equipment.from_dict(e.to_dict()) == e
assert repr(e) == "Equipment('LT-050', 'laptop', 'Asus', 'ExpertBook')"
assert str(e) == "LT-050  | laptop   | Asus ExpertBook           |   2,500.00 DT | -"
assert Employee("Karim", "Trabelsi", "Finance").full_name == "Karim Trabelsi"
inv = Inventory()
assert inv.add(e) and not inv.add(Equipment("LT-050", "laptop", "X", "Y", 2024, 1))
assert not inv.add(Equipment("TB-001", "tablet", "X", "Y", 2024, 1)) and not inv.add(Equipment("LAPTOP-1", "laptop", "X", "Y", 2024, 1))
assert len(inventory) == 3 and "pr-001" in inventory and inventory[0].asset_tag == "SV-001"
assert inventory.find("LT-001").assigned_to == "Amira Ben Salah"
assert not inventory.assign("LT-001", Employee("Karim", "Trabelsi", "Finance"))
assert [x.asset_tag for x in sorted(inventory)] == ["LT-001", "PR-001", "SV-001"]
assert [x.asset_tag for x in inventory.available()] == ["SV-001", "PR-001"]
assert inventory.total_value() == 22650.0 and inventory.stats() == {"server": 1, "laptop": 1, "printer": 1}
assert [x.asset_tag for x in inventory.older_than(5, 2026)] == ["SV-001", "PR-001"]
print("✅ Étape 5 du projet GestParc validée")

✔️ Version de référence : `scripts/gestparc/main.py`.

**Prochaine étape (S9)** : après le TP de S8, `Equipment` deviendra une classe abstraite, avec les sous-classes `Laptop`, `Server`, `Printer` et `NetworkDevice`.

---
## Pour aller plus loin (bonus)

1. **`functools.total_ordering`** : décorez `SoftwareVersion` et vérifiez que `<=`, `>` et `>=` fonctionnent avec seulement `__eq__` et `__lt__`.
2. **`__hash__`** : essayez `{SoftwareVersion("3.12")}`. Pourquoi Python refuse-t-il ? Ajoutez `__hash__` (`return hash(self.parts)`).
3. **`__iter__`** : remplacez le `__getitem__` d'`Inventory` par `def __iter__(self): return iter(self._items)` ; qu'est-ce qui fonctionne encore ?
4. Comparez votre `IPv4Address` avec le module standard `ipaddress` : `ipaddress.ip_address("192.168.1.10").is_private`.

In [ ]:
# À vous